# Phase 5 Legal Statute Identification (LSI) — Version 1
## Hierarchical Local-Global SwiGLU-LoRMA with 100 Label-Query Evidence

**Primary Objective:** Maximize legitimate held-out Macro-F1 for 100-class legal statute identification on Indian Supreme Court judgments.

### Non-Negotiable Provenance & Constraints:
1. **Strict Model Origin:** Built directly on top of the original Phase 1 `V1LegalLM` decoder backbone (113.96M parameters).
2. **Source Checkpoint:** `C:\Users\LENOVO\Downloads\v1_results\cs781_phase1_v1\final_model\v1_model.pt`.
3. **Source Tokenizer:** `d:\STORAGE\bpe_en_25k.json` (ByteLevelBPE25k, 25,000 vocabulary size).
4. **Hardware Environment:** NVIDIA GeForce RTX 4060 Laptop GPU (8 GB VRAM).
5. **Pre-Run State:** This notebook represents the verified, unexecuted implementation contract before running training.


In [ ]:
# 1. Environment Setup & Dependency Imports
import os
import sys
import math
import json
import time
import random
import hashlib
from typing import List, Dict, Tuple, Optional, Any
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score

# Ensure reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Executing on device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), '| Total VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')


In [ ]:
# 2. Load Original Phase 1 Tokenizer (bpe_en_25k.json)
BPE_PATH = r'd:\STORAGE\bpe_en_25k.json'
assert os.path.exists(BPE_PATH), f'Tokenizer file not found: {BPE_PATH}'

with open(BPE_PATH, 'r', encoding='utf-8') as f:
    bpe_data = json.load(f)

bpe_itos = bpe_data['itos']
bpe_merges = bpe_data['merges']
assert len(bpe_itos) == 25000, f'Expected 25,000 vocab tokens, got {len(bpe_itos)}'
bpe_stoi = {token: idx for idx, token in enumerate(bpe_itos)}
merge_to_id = {(bpe_stoi[m[0]], bpe_stoi[m[1]]): 261 + i for i, m in enumerate(bpe_merges)}

class ByteLevelBPE25k:
    def __init__(self, itos, stoi, merge_to_id, merges):
        self.itos = itos
        self.stoi = stoi
        self.merge_to_id = merge_to_id
        self.merges = merges
        self.vocab_size = len(itos)
        self.pad_token_id = stoi['<pad>']
        self.unk_token_id = stoi['<unk>']
        self.bos_token_id = stoi['<s>']
        self.eos_token_id = stoi['</s>']
        self.word_boundary_id = stoi['<wb>']

    def _encode_word(self, word: str) -> List[int]:
        sequence = [4 + byte for byte in word.encode('utf-8')]
        sequence.append(self.word_boundary_id)
        while len(sequence) > 1:
            best_position = None
            best_token_id = None
            for i in range(len(sequence) - 1):
                pair = (sequence[i], sequence[i + 1])
                new_token_id = self.merge_to_id.get(pair)
                if new_token_id is None: continue
                if best_token_id is None or new_token_id < best_token_id:
                    best_token_id = new_token_id
                    best_position = i
            if best_position is None: break
            sequence = sequence[:best_position] + [best_token_id] + sequence[best_position + 2:]
        return sequence

    def encode(self, text: str, add_bos: bool = False, add_eos: bool = False) -> List[int]:
        words = text.split()
        token_ids = []
        if add_bos: token_ids.append(self.bos_token_id)
        for word in words: token_ids.extend(self._encode_word(word))
        if add_eos: token_ids.append(self.eos_token_id)
        return token_ids

tokenizer = ByteLevelBPE25k(bpe_itos, bpe_stoi, merge_to_id, bpe_merges)
print('Tokenizer loaded successfully. Vocab size:', tokenizer.vocab_size)


In [ ]:
# 3. Strict Checkpoint Loading of Phase 1 Backbone
from src.models import V1LegalLM, HLGSwiGLULoRMALSI
from src.adapters import apply_lorma_to_v1

CKPT_PATH = r'C:\Users\LENOVO\Downloads\v1_results\cs781_phase1_v1\final_model\v1_model.pt'
ckpt = torch.load(CKPT_PATH, map_location='cpu', weights_only=False)
cfg = ckpt['config']
base_model = V1LegalLM(
    vocab_size=cfg['vocab_size'], n_embd=cfg['n_embd'], n_layer=cfg['n_layer'],
    n_head=cfg['n_head'], n_kv_head=cfg['n_kv_head'], max_seq_len=cfg['context_length'],
    ffn_multiplier=cfg['ffn_multiplier'], dropout=cfg['dropout']
)
load_res = base_model.load_state_dict(ckpt['model_state_dict'], strict=True)
print('Base model strictly loaded! Missing keys:', load_res.missing_keys, 'Unexpected keys:', load_res.unexpected_keys)


In [ ]:
# 4. Step-Zero Identity Verification for SwiGLU-LoRMA
from src.adapters import SwiGLULoRMALinear

test_linear = nn.Linear(768, 768, bias=False)
test_adapter = SwiGLULoRMALinear(test_linear, rank=16)
dummy_x = torch.randn(2, 32, 768)
diff = (test_adapter(dummy_x) - test_linear(dummy_x)).abs().max().item()
assert diff == 0.0, f'Step-zero identity failed with max diff {diff}!'
print(f'Step-Zero Identity verified: max absolute difference = {diff:.8f}')


In [ ]:
# 5. Full Version 1 Hierarchical Model Creation
model = HLGSwiGLULoRMALSI(
    backbone=base_model,
    num_labels=100,
    hidden_dim=768,
    lorma_rank=16,
    lorma_alpha=32.0,
    max_windows=8,
    use_label_queries=True
).to(DEVICE)

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total Model Parameters: {total_p:,} ({total_p / 1e6:.2f}M)')
print(f'Trainable Parameters:   {train_p:,} ({train_p / 1e6:.2f}M) ({train_p / total_p * 100:.2f}%)')


In [ ]:
# 6. Tiny Smoke Tests (Shape check, BCE loss, finite gradients)
model.train()
B, W, L = 2, 4, 128
dummy_w_ids = torch.randint(0, 25000, (B, W, L), device=DEVICE)
dummy_t_mask = torch.ones((B, W, L), device=DEVICE)
dummy_w_mask = torch.ones((B, W), device=DEVICE)
dummy_targets = torch.zeros((B, 100), device=DEVICE)
dummy_targets[0, [5, 12, 43]] = 1.0
dummy_targets[1, [0, 84]] = 1.0

logits = model(dummy_w_ids, dummy_t_mask, dummy_w_mask)
assert logits.shape == (B, 100), f'Expected shape (2, 100), got {logits.shape}'

criterion = nn.BCEWithLogitsLoss()
loss = criterion(logits, dummy_targets)
loss.backward()

for name, p in model.named_parameters():
    if p.requires_grad:
        assert p.grad is not None, f'Missing gradient for {name}'
        assert torch.isfinite(p.grad).all(), f'Infinite/NaN gradient in {name}'

print(f'Smoke test passed: loss = {loss.item():.4f}, all gradients finite and valid!')


In [ ]:
# 7. Dataset Loading and Execution Plan
from src.dataset import LSIDataset, collate_lsi_batch
from src.train import LSITrainer

TRAIN_PATH = r'C:\Users\LENOVO\Downloads\train (2).json'
DEV_PATH = r'C:\Users\LENOVO\Downloads\dev (1).json'

with open(TRAIN_PATH, 'r', encoding='utf-8') as f: train_records = json.load(f)
with open(DEV_PATH, 'r', encoding='utf-8') as f: dev_records = json.load(f)

print(f'Loaded {len(train_records):,} train records and {len(dev_records):,} dev records.')
